# Ride-Sharing Demand & Revenue Intelligence



## Business Objective


- The goal of this project is to analyze NYC Yellow Taxi trip data (August 2026) to understand demand patterns, revenue distribution, and travel behavior, and prepare the dataset for a dynamic surge pricing simulation.


### This Notebook Focuses On:


-.Data Cleaning — Handling missing values and invalid records.

.Outlier Analysis — Identifying unusual trip distances, fares, and durations.

.Feature Engineering — Creating new columns for meaningful analysis.

.Data Preparation — Preparing a clean dataset for SQL, Power BI, and Machine Learning.

## Data Cleaning & Feature Engineering

### Import libraries

In [13]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")


Libraries imported successfully!


### Dataset loading

###
**Dataset:** NYC Yellow Taxi Trip Records

**Period:** August 2026

**Source:** NYC Taxi & Limousine Commission

**Format:** Parquet

We load the raw dataset to understand its structure
before applying any cleaning operations.


## Dataset Loading............

In [15]:

file_path = Path("yellow_tripdata_2026-08.parquet")

print("Dataset found:", file_path.exists())

df = pd.read_parquet(file_path)

df.head()


Dataset found: True


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-08-01 00:03:04,2026-08-01 00:07:51,1.0,0.60,1.0,N,140,262,1,6.5,3.5,0.5,2.30,0.00,1.0,13.80,2.5,0.0,0.00,None
1,1,2026-08-01 00:38:45,2026-08-01 00:44:02,1.0,0.70,1.0,N,140,262,1,6.5,3.5,0.5,3.45,0.00,1.0,14.95,2.5,0.0,0.00,None
2,1,2026-08-01 00:54:51,2026-08-01 01:01:07,1.0,1.10,1.0,N,140,263,1,7.9,3.5,0.5,2.60,0.00,1.0,15.50,2.5,0.0,0.00,None
3,2,2026-08-01 00:27:35,2026-08-01 00:54:27,1.0,5.24,1.0,N,164,80,1,29.6,1.0,0.5,8.56,7.46,1.0,51.37,2.5,0.0,0.75,None
4,2,2026-08-01 00:39:08,2026-08-01 01:00:23,1.0,3.57,1.0,N,148,141,1,20.5,1.0,0.5,5.25,0.00,1.0,31.50,2.5,0.0,0.75,None


## Exploratory Data Analysis (EDA)

In [16]:

print("Shape:", df.shape)
print()

df.info(show_counts=True)


Shape: (3336716, 21)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3336716 entries, 0 to 3336715
Data columns (total 21 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               3336716 non-null  int32         
 1   tpep_pickup_datetime   3336716 non-null  datetime64[us]
 2   tpep_dropoff_datetime  3336716 non-null  datetime64[us]
 3   passenger_count        2415535 non-null  float64       
 4   trip_distance          3336716 non-null  float64       
 5   RatecodeID             2415535 non-null  float64       
 6   store_and_fwd_flag     2415535 non-null  object        
 7   PULocationID           3336716 non-null  int32         
 8   DOLocationID           3336716 non-null  int32         
 9   payment_type           3336716 non-null  int64         
 10  fare_amount            3336716 non-null  float64       
 11  extra                  3336716 non-null  float64       
 12  mta_ta

## Missing values

In [17]:

print("Missing values:")

df.isnull().sum()


Missing values:


VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count           921181
trip_distance                  0
RatecodeID                921181
store_and_fwd_flag        921181
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge      921181
Airport_fee               921181
cbd_congestion_fee             0
request_source           2415867
dtype: int64

## Missing percentages

In [18]:

print("Percentage of missing values:")

print(
    round(
        (df.isnull().sum() / len(df)) * 100,
        2
    )
)


Percentage of missing values:
VendorID                  0.00
tpep_pickup_datetime      0.00
tpep_dropoff_datetime     0.00
passenger_count          27.61
trip_distance             0.00
RatecodeID               27.61
store_and_fwd_flag       27.61
PULocationID              0.00
DOLocationID              0.00
payment_type              0.00
fare_amount               0.00
extra                     0.00
mta_tax                   0.00
tip_amount                0.00
tolls_amount              0.00
improvement_surcharge     0.00
total_amount              0.00
congestion_surcharge     27.61
Airport_fee              27.61
cbd_congestion_fee        0.00
request_source           72.40
dtype: float64


## Columns containing NULL values

In [19]:

print("Columns that have null values:")

df.columns[df.isnull().any()].tolist()


Columns that have null values:


['passenger_count',
 'RatecodeID',
 'store_and_fwd_flag',
 'congestion_surcharge',
 'Airport_fee',
 'request_source']

## Select columns in aug-2026

In [20]:

columns_to_keep = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

available_columns = [
    col for col in columns_to_keep
    if col in df.columns
]

missing_columns = [
    col for col in columns_to_keep
    if col not in df.columns
]

print("Unavailable columns:", missing_columns)

df_1 = df[available_columns].copy()

df_1.head()


Unavailable columns: []


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2026-08-01 00:03:04,2026-08-01 00:07:51,1.0,0.60,1.0,140,262,1,6.5,3.5,0.5,2.30,0.00,1.0,13.80,2.5,0.0,0.00
1,1,2026-08-01 00:38:45,2026-08-01 00:44:02,1.0,0.70,1.0,140,262,1,6.5,3.5,0.5,3.45,0.00,1.0,14.95,2.5,0.0,0.00
2,1,2026-08-01 00:54:51,2026-08-01 01:01:07,1.0,1.10,1.0,140,263,1,7.9,3.5,0.5,2.60,0.00,1.0,15.50,2.5,0.0,0.00
3,2,2026-08-01 00:27:35,2026-08-01 00:54:27,1.0,5.24,1.0,164,80,1,29.6,1.0,0.5,8.56,7.46,1.0,51.37,2.5,0.0,0.75
4,2,2026-08-01 00:39:08,2026-08-01 01:00:23,1.0,3.57,1.0,148,141,1,20.5,1.0,0.5,5.25,0.00,1.0,31.50,2.5,0.0,0.75


## Check selected columns

In [21]:

df_1.info(show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3336716 entries, 0 to 3336715
Data columns (total 19 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               3336716 non-null  int32         
 1   tpep_pickup_datetime   3336716 non-null  datetime64[us]
 2   tpep_dropoff_datetime  3336716 non-null  datetime64[us]
 3   passenger_count        2415535 non-null  float64       
 4   trip_distance          3336716 non-null  float64       
 5   RatecodeID             2415535 non-null  float64       
 6   PULocationID           3336716 non-null  int32         
 7   DOLocationID           3336716 non-null  int32         
 8   payment_type           3336716 non-null  int64         
 9   fare_amount            3336716 non-null  float64       
 10  extra                  3336716 non-null  float64       
 11  mta_tax                3336716 non-null  float64       
 12  tip_amount             33367

## Investigate missing passenger counts

In [22]:

print("Missing passenger counts:")
print(df_1["passenger_count"].isnull().sum())

print("\nPassenger count distribution:")
print(
    df_1["passenger_count"].value_counts(
        dropna=False
    ).head(15)
)


Missing passenger counts:
921181

Passenger count distribution:
passenger_count
1.0    1965795
NaN     921181
2.0     301322
3.0      76385
4.0      57681
0.0       7478
5.0       4264
6.0       2606
8.0          4
Name: count, dtype: int64


## Check invalid records

In [23]:

df_1["tpep_pickup_datetime"] = pd.to_datetime(
    df_1["tpep_pickup_datetime"],
    errors="coerce"
)

df_1["tpep_dropoff_datetime"] = pd.to_datetime(
    df_1["tpep_dropoff_datetime"],
    errors="coerce"
)

df_1["trip_duration_min"] = (
    df_1["tpep_dropoff_datetime"]
    - df_1["tpep_pickup_datetime"]
).dt.total_seconds() / 60

invalid_summary = {
    "Zero or negative distance":
        (df_1["trip_distance"] <= 0).sum(),

    "Zero or negative fare":
        (df_1["fare_amount"] <= 0).sum(),

    "Zero or negative total":
        (df_1["total_amount"] <= 0).sum(),

    "Invalid duration":
        (df_1["trip_duration_min"] <= 0).sum(),

    "Missing timestamps":
        df_1[
            ["tpep_pickup_datetime",
             "tpep_dropoff_datetime"]
        ].isnull().any(axis=1).sum(),

    "Outside August 2026":
        (
            (df_1["tpep_pickup_datetime"] < "2026-08-01")
            | (df_1["tpep_pickup_datetime"] >= "2026-09-01")
        ).sum()
}

display(pd.Series(invalid_summary))


Zero or negative distance    119059
Zero or negative fare         17249
Zero or negative total        15827
Invalid duration              43185
Missing timestamps                0
Outside August 2026              15
dtype: int64

## Filter invalid records

In [24]:

raw_count = len(df_1)

valid_mask = (
    df_1["tpep_pickup_datetime"].ge("2026-08-01")
    & df_1["tpep_pickup_datetime"].lt("2026-09-01")
    & df_1["trip_distance"].gt(0)
    & df_1["fare_amount"].gt(0)
    & df_1["total_amount"].gt(0)
    & df_1["trip_duration_min"].gt(0)
)

df_1 = df_1.loc[valid_mask].copy()

print("Records before cleaning:", raw_count)
print("Records after filtering:", len(df_1))
print("Records excluded:", raw_count - len(df_1))


Records before cleaning: 3336716
Records after filtering: 3163242
Records excluded: 173474



## Feature Extraction

Create new columns to understand:

- Pickup hour
- Pickup day
- Day of the week
- Weekday vs weekend demand
- Trip duration


## Create date and time features

In [25]:

df_1["pickup_hour"] = (
    df_1["tpep_pickup_datetime"].dt.hour
)

df_1["pickup_day"] = (
    df_1["tpep_pickup_datetime"].dt.day
)

df_1["pickup_weekday"] = (
    df_1["tpep_pickup_datetime"].dt.weekday
)

df_1["is_weekend"] = (
    df_1["pickup_weekday"]
    .isin([5, 6])
    .astype(int)
)

df_1.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_hour,pickup_day,pickup_weekday,is_weekend
0,1,2026-08-01 00:03:04,2026-08-01 00:07:51,1.0,0.60,1.0,140,262,1,6.5,3.5,0.5,2.30,0.00,1.0,13.80,2.5,0.0,0.00,4.783333,0,1,5,1
1,1,2026-08-01 00:38:45,2026-08-01 00:44:02,1.0,0.70,1.0,140,262,1,6.5,3.5,0.5,3.45,0.00,1.0,14.95,2.5,0.0,0.00,5.283333,0,1,5,1
2,1,2026-08-01 00:54:51,2026-08-01 01:01:07,1.0,1.10,1.0,140,263,1,7.9,3.5,0.5,2.60,0.00,1.0,15.50,2.5,0.0,0.00,6.266667,0,1,5,1
3,2,2026-08-01 00:27:35,2026-08-01 00:54:27,1.0,5.24,1.0,164,80,1,29.6,1.0,0.5,8.56,7.46,1.0,51.37,2.5,0.0,0.75,26.866667,0,1,5,1
4,2,2026-08-01 00:39:08,2026-08-01 01:00:23,1.0,3.57,1.0,148,141,1,20.5,1.0,0.5,5.25,0.00,1.0,31.50,2.5,0.0,0.75,21.250000,0,1,5,1


## Outlier analysis

### Removing Outliers

Investigate unusually high and low values using
the 1st and 99th percentiles.

The percentile limits help identify unusual records.
They are not automatic proof that those records
are incorrect.

Review the findings before removing outliers.


## Calculate outlier thresholds

In [27]:

outlier_columns = [
    "trip_distance",
    "fare_amount",
    "total_amount",
    "trip_duration_min"
]

display(
    df_1[outlier_columns].describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
    )
)

for col in outlier_columns:
    low = df_1[col].quantile(0.01)
    high = df_1[col].quantile(0.99)

    unusual_count = (
        (df_1[col] < low)
        | (df_1[col] > high)
    ).sum()

    print(
        f"{col}: Lower={low:.2f}, "
        f"Upper={high:.2f}, "
        f"Outside range={unusual_count}"
    )


,trip_distance,fare_amount,total_amount,trip_duration_min
count,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06
mean,5.118872e+00,2.113321e+01,3.016676e+01,1.747609e+01
std,4.319269e+02,1.793047e+01,2.188965e+01,2.389789e+01
min,1.000000e-02,1.000000e-02,1.010000e+00,1.666667e-02
1%,1.900000e-01,4.400000e+00,9.800000e+00,2.133333e+00
25%,1.150000e+00,1.000000e+01,1.745000e+01,8.766667e+00
50%,2.090000e+00,1.560000e+01,2.355000e+01,1.433333e+01
75%,4.400000e+00,2.572000e+01,3.396000e+01,2.220000e+01
99%,1.974000e+01,8.070000e+01,1.052600e+02,6.191667e+01
max,2.231987e+05,1.860800e+03,1.884550e+03,8.373667e+03


trip_distance: Lower=0.19, Upper=19.74, Outside range=62737
fare_amount: Lower=4.40, Upper=80.70, Outside range=49064
total_amount: Lower=9.80, Upper=105.26, Outside range=60816
trip_duration_min: Lower=2.13, Upper=61.92, Outside range=62906



## Calculating Revenue Metrics

Create:

- Recorded charges per mile
- Recorded charges per minute
- Recorded tip percentage

For this project, total_amount represents recorded
passenger charges, not business profit.


## Create financial features

In [28]:

df_1["revenue_per_mile"] = (
    df_1["total_amount"] / df_1["trip_distance"]
).round(2)

df_1["revenue_per_minute"] = (
    df_1["total_amount"] / df_1["trip_duration_min"]
).round(2)

df_1["tip_percentage"] = (
    df_1["tip_amount"] / df_1["fare_amount"]
).round(4)

df_1.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_hour,pickup_day,pickup_weekday,is_weekend,revenue_per_mile,revenue_per_minute,tip_percentage
0,1,2026-08-01 00:03:04,2026-08-01 00:07:51,1.0,0.60,1.0,140,262,1,6.5,3.5,0.5,2.30,0.00,1.0,13.80,2.5,0.0,0.00,4.783333,0,1,5,1,23.00,2.89,0.3538
1,1,2026-08-01 00:38:45,2026-08-01 00:44:02,1.0,0.70,1.0,140,262,1,6.5,3.5,0.5,3.45,0.00,1.0,14.95,2.5,0.0,0.00,5.283333,0,1,5,1,21.36,2.83,0.5308
2,1,2026-08-01 00:54:51,2026-08-01 01:01:07,1.0,1.10,1.0,140,263,1,7.9,3.5,0.5,2.60,0.00,1.0,15.50,2.5,0.0,0.00,6.266667,0,1,5,1,14.09,2.47,0.3291
3,2,2026-08-01 00:27:35,2026-08-01 00:54:27,1.0,5.24,1.0,164,80,1,29.6,1.0,0.5,8.56,7.46,1.0,51.37,2.5,0.0,0.75,26.866667,0,1,5,1,9.80,1.91,0.2892
4,2,2026-08-01 00:39:08,2026-08-01 01:00:23,1.0,3.57,1.0,148,141,1,20.5,1.0,0.5,5.25,0.00,1.0,31.50,2.5,0.0,0.75,21.250000,0,1,5,1,8.82,1.48,0.2561


## Final checks

In [29]:

print("Final shape:", df_1.shape)

print("\nRemaining missing values:")
display(df_1.isnull().sum())

print("\nStatistical summary:")
display(df_1.describe())

print("\nInvalid distance records:",
      (df_1["trip_distance"] <= 0).sum())

print("Invalid duration records:",
      (df_1["trip_duration_min"] <= 0).sum())


Final shape: (3163242, 27)

Remaining missing values:


VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          838431
trip_distance                 0
RatecodeID               838431
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     838431
Airport_fee              838431
cbd_congestion_fee            0
trip_duration_min             0
pickup_hour                   0
pickup_day                    0
pickup_weekday                0
is_weekend                    0
revenue_per_mile              0
revenue_per_minute            0
tip_percentage                0
dtype: int64


Statistical summary:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_hour,pickup_day,pickup_weekday,is_weekend,revenue_per_mile,revenue_per_minute,tip_percentage
count,3.163242e+06,3163242,3163242,2.324811e+06,3.163242e+06,2.324811e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,2.324811e+06,2.324811e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06,3.163242e+06
mean,1.828618e+00,2026-08-16 05:02:56.592275,2026-08-16 05:20:25.157753,1.266888e+00,5.118872e+00,4.148987e+00,1.603452e+02,1.593908e+02,8.478033e-01,2.113321e+01,1.195067e+00,4.948319e-01,2.791731e+00,6.183496e-01,9.986392e-01,3.016676e+01,2.292609e+00,1.996059e-01,5.785287e-01,1.747609e+01,1.396936e+01,1.560749e+01,3.169174e+00,3.366192e-01,3.313225e+01,3.394869e+00,1.613068e-01
min,1.000000e+00,2026-08-01 00:00:00,2026-08-01 00:02:40,0.000000e+00,1.000000e-02,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e-02,-7.500000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.010000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.666667e-02,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.000000e+00,2026-08-08 04:18:56.500000,2026-08-08 04:36:51,1.000000e+00,1.150000e+00,1.000000e+00,1.140000e+02,1.070000e+02,0.000000e+00,1.000000e+01,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.745000e+01,2.500000e+00,0.000000e+00,7.500000e-01,8.766667e+00,1.000000e+01,8.000000e+00,1.000000e+00,0.000000e+00,7.450000e+00,1.460000e+00,0.000000e+00
50%,2.000000e+00,2026-08-15 21:03:18.500000,2026-08-15 21:21:30,1.000000e+00,2.090000e+00,1.000000e+00,1.610000e+02,1.610000e+02,1.000000e+00,1.560000e+01,0.000000e+00,5.000000e-01,2.000000e+00,0.000000e+00,1.000000e+00,2.355000e+01,2.500000e+00,0.000000e+00,7.500000e-01,1.433333e+01,1.500000e+01,1.500000e+01,3.000000e+00,0.000000e+00,1.101000e+01,1.770000e+00,1.691000e-01
75%,2.000000e+00,2026-08-23 18:49:17,2026-08-23 19:07:06,1.000000e+00,4.400000e+00,1.000000e+00,2.310000e+02,2.330000e+02,1.000000e+00,2.572000e+01,2.500000e+00,5.000000e-01,3.930000e+00,0.000000e+00,1.000000e+00,3.396000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2.220000e+01,1.900000e+01,2.300000e+01,5.000000e+00,1.000000e+00,1.580000e+01,2.250000e+00,2.851000e-01
max,6.000000e+00,2026-08-31 23:59:59,2026-09-01 20:16:00,8.000000e+00,2.231987e+05,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,1.860800e+03,2.325000e+01,1.150000e+01,4.444400e+02,1.064600e+02,1.000000e+00,1.884550e+03,2.750000e+00,2.000000e+00,7.500000e-01,8.373667e+03,2.300000e+01,3.100000e+01,6.000000e+00,1.000000e+00,8.042500e+04,8.042500e+03,3.333000e+01
std,4.219142e-01,NaN,NaN,6.687689e-01,4.319269e+02,1.704772e+01,6.639237e+01,7.077195e+01,6.156184e-01,1.793047e+01,1.785175e+00,5.249938e-02,3.943523e+00,2.325409e+00,3.182597e-02,2.188965e+01,6.894685e-01,5.994743e-01,3.149621e-01,2.389789e+01,6.215982e+00,9.005187e+00,2.024708e+00,4.725535e-01,3.118644e+02,3.975006e+01,1.813858e-01



Invalid distance records: 0
Invalid duration records: 0


## Save cleaned Parquet

In [30]:

processed_dir = file_path.parent.parent / "Processed"

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    processed_dir
    / "Cleaned_yellow_tripdata_2026-08.parquet"
)

df_1.to_parquet(
    output_path,
    index=False
)

print("Saved successfully:", output_path)


Saved successfully: Processed\Cleaned_yellow_tripdata_2026-08.parquet
